# Automatische Erkennung von Figuren mit *InsightFace*

Dieses Jupyter Notebook annotiert das Auftreten von Figuren in einer Videodateien mithilfe von **InsightFace**. Dabei werden zunächst Gesichter erkannt (*face detetction*) und darauf aufbauen auf der Grundlage von Referenzabbildungen entsprechende Figuren (*face recognition*). Als Ergebnis entstehen:

- eine Tabelle im `.csv`-Format mit Namen der Videodatei, Namen der erkannten Figuren, Start und Endzeit der Erkennung einer Figur in Frames, Start und Endzeit der Erkennung einer Figur in Sekunden, Start und Endzeit der Erkennung einer Figur in Stunden:Minuten:Sekunden, 
- eine Tabelle im `.csv`-Format mit der Zeit in Sekunden, der Zeit in Stunden:Minuten:Sekunden, dem durchschnittlichen Skalarprodukt (**siehe Kapitel xx 4.8?**), dem durchschnittlichen predicition-score (**siehe Kapitel xx 4.8?**) und dem Namen der Figur 
- eine weitere Tabelle im `.csv`-Format zu den Metadaten der Analyse (**vgl. Kapitel XXX 4.8??**),
- eine ELAN-Datei (`.eaf`) zum Import und zur Sichtung der Annotationsdaten in *VIAN* oder anderen Programmen zur Videoannotation (**vgl. Kapitel xxx**).

Der Arbeitsablauf ist in vier Abschnitte gegliedert: Festlegen der Parameter, technische Vorbereitung, Analyse bzw. Anwendung der jeweils passenden Erkennungsmodelle und Export der Ergebnisse. Die Texte zwischen den Codeblöcken erläutern vor allem den Zweck dieser einzelnen Schritte und die Bedeutung der erzeugten Dateien, während die Kommentare innerhalb des Codes auf Englisch dessen Funktionsweise vermitteln.

Wir empfehlen bei der Bearbeitung dieses Kapitels in Form eines Jupyter Notebooks die Codeblöcke einzeln nacheinander auszuführen und nicht mit Funktionen wie "Alle ausführen" in Colab oder "Run All Cells" in anderen Anwendungen zur Ausführung von Jupyter Notebooks. Die separate Ausführung der Codeblöcke erleichtert das Verständnis der mit ihnen verbundenen Funktionen und gewährleistet, dass alle vorbereitenden Schritte für nachfolgende Codeblöcke bereits durchlaufen wurden.

## Festlegen der Parameter

In diesem Abschnitt werden die zentralen Parameter für die spätere Analyse festgelegt. Sie können an dieser Stelle angepasst werden, ohne Änderungen im weiteren Code vornehmen zu müssen. Die dafür verwendeten Variablen werden alle großgeschrieben, um sie von den späteren Variablen zur Strukturierung und Erklärung des Codes abzugrenzen. Zum Erlernen der Funktionen des Codes in diesem Jupyter Notebook und für einen ersten Probelauf empfiehlt es sich, die voreingestellten Werte beizubehalten. 

Zuerst werden die Ordner für die zu analysierenden Videodateien (`VIDEO_PATH`) und den Export der Annotationsergebnisse (`EXPORT_PATH`) festgelegt. Sollten die Ordner noch nicht existieren, werden sie mit entsprechender Benennung beim Ausführen des Codeblocks automatisch erstellt.

Im Video-Ordner wird die zu analysierende Videodatei abgelegt. Im gleichen Ordner werden auch die entsprechenden *Referenzabbildungen* für die Figurenerkennung abgespeichert (**siehe Kapitel xx 4.8??**). Für jede zu erkennende Figur muss mindestens eine Referenzabbildung hinterlegt werden, mehrere Abbildungen einer Figur, z.B. aus verschiedenen Kamerawinkeln, erhöhen die Wahrscheinlichkeit einer korrekten Erkennung. Für den Export der `.eaf`-Datei muss zusätzlich die Annotationsdatei zu den erkannten Einstellungen aus dem Jupyter Notebook **Automatische Erkennung von Einstellungen mit *TransNetV2*** im Ordner gespeichert werden.

Aktuell wird nur eine Videodatei pro Ordner unterstützt. Falls mehrere Videos in einem Ordner abgelegt werden, wird beim Ausführen des Notebooks darauf hingewiesen. Für jede zu analysierende Videodatei sollte im Video-Ordner ein eigener Unterordner angelegt werden. Das bedeutet auch, dass für jedes Video eigene Referenzbilder im gleichen Ordner, in der auch die Videodatei liegt, hinzugefügt werden müssen und Videos sich keine Referenzbilder "teilen" können. Hinsichtlich der Anzahl an Ordnern gibt es keine Beschränkungen. Videodateien in unterschiedlichen Ordern können dieselben Namen haben. Dieser muss zudem nicht mit dem Ordnernamen identisch sein. 

In diesem Ordner werden auch die für diese *OER* vorbereiteten Testdateien im zip-Format abgelegt. Die Testdateien beinhalten neben dem zu analysierenden Trailer die passenden Referenzabbildungen und die Annotationsdatei zur Einstellungserkennung. Sie sind für die Ausführung dieses Jupyter Notebooks zu Lernzwecken vorgesehen und wir empfehlen, diese hierfür auch zu verwenden. Ist deren Download nicht erwünscht, kann die Variable `DOWNLOAD_TESTFILES` auf `False` gesetzt werden. In diesem Fall müssen eigene Videodateien in den Ordner "videos" kopiert werden.

Des Weiteren werden zwei Schwellenwerte festgelegt. Zur Bedeutung von Schwellenwerten vgl. auch **Kapitel xx**. Der durch `FACE_DETECTION_THRESHOLD` festgelegte Schwellenwert muss während der Gesichtserkennung (*face detection*) überschritten werden, damit das erkannte Gesicht weiter verarbeitet wird. Im Fall von *Insightface* und dem Ziel der Erkennung weniger prominenter Figuren ist dieser Wert nicht von erheblicher Relevanz, da die entsprechenden Gesichter i.d.R. mit einer hohen Wahrscheinlichkeit von 0.8 oder mehr erkannt werden. **ToDo: nicht genau klar was mit *weniger prominenter Figuren* gemeint ist.** `CHARACTER_RECOGNITION_THRESHOLD` legt den Schwellenwert fest der überschritten werden muss, damit eine Figur erkannt wird (*face recognition*) und einem entsprechenden Frame in der Videodatei zugeordnet wird. Der zu prüfende Wert beruht auf dem Skalarprodukt zwischen den Embeddings (**siehe Kapitel xx 4,8?**). Auch hier ist der Wert von geringerer Relevanz, da auch die weniger genauen *Insightface* Modell-Pakete wie *buffalo_s* (**siehe Kapitel XX**) bei korrekter Erkennung im Durchschnitt einen hohen Ähnlichkeitswert von  mehr als 0.75 erzeugen.

`SKIP_FRAMES` gibt die Anzahl an Frames an, die nach nach der Auswertung eines Frames übersprungen werden. Bei einem Wert von 0 werden keine Frames übersprungen, jedoch jedoch benötigt die Auswertung aller Frames einer Videodatei eine längere Analysezeit. Je höher der Wert von `Skip Frames` gesetzt wird, desto weniger Frames müssen insgesamt analysiert werden und die benötigte Rechenzeit für die Figurenerkennung nimmt entsprechend ab (**vgl. Kapitel xx technische Voraussetzungen??**). Allerdings kann dadurch auch die Genauigkeit der Erkennungsleistung abnehmen.

Mit `TOLERANCE_FRAMES` wird die Anzahl an fehlenden Frames festgelegt, über die hinweg Frames zu einer langen Annotation zusammengefügt werden. Einzelne Frames können z.B. bei fehlerhaften Erkennungen durch ungünstige Kamerawinkel oder schlechter Bildqualität fehlen; möglich ist auch, dass Figuren im Bild bei der Erkennung durch mit `SKIP_FRAMES` übersprungene Frames nicht erfasst wurden. `TOLERANCE_FRAMES` gewährleistet also, dass in diesen Fällen dennoch eine Annotation für die Figur erstellt wird.

Wenn `DOWNLOAD_TESTFILES` auf `True` gesetzt ist, werden die vorbereiteten Trailer und Screenshots heruntergeladen. Um eigene Filme zu verwenden, sollte dieser Parameter daher auf `False` gestellt werden. Eigene Videodateien und Referenzabbildungen müssen in den bei `VIDEO_PATH` festgelegten Ordner abgelegt werden. In unserem Beispiel ist dies der Ordner "video_face".

Es können auch eigene ZIP-Archive von Online-Speicherplätzen heruntergeladen werden. Hierzu muss ein entsprechender Download-Link innerhalb von Anführungszeichen der Variable `CUSTOM_ZIP_LINK` zugeordnet werden, z.B. `CUSTOM_ZIP_LINK = "https://example.org/video_archive.zip"`. Wichtig ist dabei, dass es sich um einen Direktlink zur `.zip`-Datei handelt, nicht um einen Link zu einer Website oder einer Datei in einem anderen Format. Außerdem sollte beim Download mehrerer Archive nacheinander jeweils die Variable `CUSTOM_ARCHIVE_NAME` geändert werden, da Archive mit demselben Namen nicht erneut heruntergeladen werden. Die Option zum Download eigener Dateien ist besonders bei der Analyse von Langfilmen in *Google Colab* hilfreich, da die Videodateien dadurch nicht manuell und unter erheblichem Zeitaufwand in die *Colab*-Umgebung hochgeladen werden müssen. Der Download von einem Online-Speicher wie z.B. einem Cloud-Dienst benötigt in der Regel weniger Zeit.

Mit `DIVIDER` wird die Art der Trennlinie zwischen den Ausgaben des Jupyter Notebooks festgelegt.


In [1]:
import os

VIDEO_PATH = "videos_face"
os.makedirs(VIDEO_PATH, exist_ok=True)
EXPORT_PATH = "export_face"
os.makedirs(EXPORT_PATH, exist_ok=True)

# Settings
FACE_DETECTION_THRESHOLD = 0.3
CHARACTER_RECOGNITION_THRESHOLD = 0.3
SKIP_FRAMES = 4

TOLERANCE_FRAMES = 25

# Testfile settings
DOWNLOAD_TESTFILES = True

# Add your own custom link inside quotes to download additional files into the video folder.
CUSTOM_ZIP_LINK = "insert link here"

# Change the name of the custom archives. A new archive with the same name will not be downloaded. 
CUSTOM_ARCHIVE_NAME = "custom_archive"

# Line DIVIDER
DIVIDER = "─" * 200

---
## Technische Vorbereitung

Bevor die Videos analysiert werden können, müssen die benötigten Softwarepakete installiert, in das Notebook eingebunden und das vortrainierte Insightface-Modell geladen werden.

### Installation der Programmbibliotheken

Der folgende Codeblock installiert externe Programmbibliotheken, die nicht in der Standardinstallation von *Python* enthalten sind. Dazu zählen neben dem Modell `Insightface` zur Gesichtserkennung
- `pandas` zum Bearbeiten und Exportieren von tabellarischen Daten,
- `opencv-python` zur Videobearbeitung,
- `numpy` zum wissenschaftliches Rechnen (Umgang mit Vektoren etc.)
- `pympi-ling` zum Schreiben von *ELAN*-Dateien (`.eaf`).
- `tqdm` für die prozentuale Anzeige des Bearbeitungsfortschritts durch einen Balken
- `torch` für den Einsatz von Machine Learning
- `onnxruntime` bzw `onnxruntime-gpu` zur Ausführung von Machine-Learning-Modellen

Für die wichtigsten Bibliotheken wird eine spezifische Version installiert, um deren interdependente Funktionsweise zu garantieren.

Im Zuge der Installation der Bibliotheken wird automatisch ermittelt, ob eine Cuda-fähige GPU vorhanden ist und die die Installation entsprechend angepasst. Mit einer Cuda-fähigen GPU (Graphics Processing Unit) können Rechenoperationen wesentlich schneller ausgeführt werden als mit einer 'normalen' CPU (Central Processing Unit) und damit die Zeit für die Annotation einer Videodatei erheblich verkürzt werden. Zum genaueren Verständnis des Unterschieds von GPU und CPU **siehe Kapitel xxx**. Auf die Information, ob eine GPU oder eine CPU zur Verfügung steht wird später erneut Bezug genommen, um die jeweils geeigneten Modell-Pakete automatisch auszuwählen.

In einer lokalen und bei einer erneuten Ausführung des Notebooks identischen Arbeitsumgebung muss diese Zelle nicht bei jeder Sitzung nochmals ausgeführt werden. In *Google Colab* ist eine erneute Installation nach dem Neustart der Laufzeit hingegen notwendig, da alle speziell installierten Programmbibliotheken beim Beenden der Laufzeit gelöscht werden.

In [2]:
print("Installing dependencies.")
import sys

try:
    from google.colab import files
    colab = True
except Exception as e:
    print("Using non-colab environment.")
    colab = False

# Use code for Windows or Unix systems.
if sys.platform.startswith("win"):
    # Suppress pip version checks.
    !pip -q config set global.disable-pip-version-check true > NUL 2>&1
    # Check for Cuda-gpu with a termin command.
    cuda = (os.system("nvidia-smi > NUL 2>&1") == 0)
else:
    # Suppress pip version checks.
    !pip -q config set global.disable-pip-version-check true > /dev/null 2>&1
    # Check for Cuda-gpu with a termin command.
    cuda = (os.system("nvidia-smi > /dev/null 2>&1") == 0)

# Install common dependencies
!pip install -q pandas
!pip install -q opencv-python==4.13.0.92
!pip install -q numpy
!pip install -q pympi-ling==1.71
!pip install -q tqdm==4.68.3

if cuda:
    print("GPU detected. Installing GPU backend.")

    # Install GPU backend.
    if not colab:
        !pip install -q torch --index-url https://download.pytorch.org/whl/cu126

    !pip install -q "onnxruntime-gpu==1.22.0"
    
    # Double check afterwards if Cuda is available.
    import onnxruntime as ort
    print(ort.get_available_providers())
    if "CUDAExecutionProvider" in ort.get_available_providers():
        print("Using GPU backend.")
        !nvidia-smi --query-gpu=name --format=csv,noheader
        ort.preload_dlls(cuda=True, cudnn=True, msvc=True)
    else:
        print("CUDA provider unavailable. Falling back to CPU.")
        cuda = False

if not cuda:
    print("Using CPU backend.")

    # Install CPU backend.
    !pip install -q "onnxruntime==1.22.0"

# Install Insightface
!pip install -q insightface==1.0.1

Installing dependencies.
Using non-colab environment.


Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


Using CPU backend.


Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


### Import der Programmbibliotheken

Durch den Import werden die installierten Funktionen nun in der aktuellen Notebook-Sitzung verfügbar. Die Bibliotheken übernehmen unterschiedliche Aufgaben, die im Codeblock jeweils kurz erläutert werden.

In [3]:
import insightface
import pandas as pd
from pathlib import Path
import cv2
import numpy as np
import time
import datetime
from tqdm.notebook import tqdm
from urllib.request import urlretrieve
import zipfile
import pympi
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
print("Dependencies imported.")


Dependencies imported.


### Download des Analysematerials
Zum Download des Testmaterials und eines eigenen `.zip`-Archivs wird eine wiederverwendbare Funktion definiert und direkt angewendet. 

Falls bei der [Festlegung der Parameter](#festlegen-der-parameter) `DOWNLOAD_TESTFILES = True` gesetzt ist, lädt das Notebook das `.zip`-Archiv `shotDetection_testfiles.zip` (**ToDo: sollte `faceDetection_testfiles.zip` heißen????**) herunter, das einen Trailer zu den Konrad Wolfs Film **XX**, entsprechende Referenzabbildungen zu den Figuren und eine csv-Datei mit den Einstellungs-Annotationen zum Trailer enthält. Diese werden für die Durchführung der Figurenerkennung und die Ausgabe der Annotations-Dateien benötigt. **[CHANGE THIS]**. Liegt `shotDetection_testfiles.zip` aufgrund einer vorherigen Ausführung bereits auf derselben Ordnerebene wie das Notebook, wird die Datei nicht erneut heruntergeladen. Anschließend wird ihr Inhalt in den zuvor festgelegten Videoordner entpackt. Zu Lernzwecken im Rahmen der OER empfehlen wir den Download und die Arbeit mit den Testdateien im Notebook.

Dieser Schritt ist für die Arbeit mit eigenen Video- und Film-Korpora nicht erforderlich. Eigene Videodateien, Referenzabbildungen für die Erkennung der Figuren und Einstellungsannotationen zu den Videodateien können direkt im `VIDEO_PATH` Ordner oder in dessen Unterordnern abgelegt werden. In diesem Fall muss `DOWNLOAD_TESTFILES = False` gesetzt werden. Außerdem kann mittels des Parameters `CUSTOM_ZIP_LINK` ein eigenes `.zip`-Archiv heruntergeladen werden, wie bereits eingangs im Kontext der [Festlegung der Parameter](#festlegen-der-parameter) erläutert. In beiden Fällen sollte der Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) bereits ausgeführt worden sein, um falls notwendig den Videoordner erst zu erstellen.

In [4]:
# Define a reusable function for downloading .zip archives.
# This function takes three arguments: The download link, the name for the downloaded archive, and the directory to unzip the archive into.
def download_zip(url: str, archive_name: str, target_directory: str):

    # Add .zip to the filename and turn it into a relative path in the working directory.
    archive_path = Path(f"{archive_name}.zip")

    # Download only if the archive is not already present in the working directory.
    if not archive_path.is_file():
        # Download the file.
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
        print(f"{archive_name} downloaded.")
    else:
        print(f"{archive_name} already downloaded.")

    # Unzip the file to the target_directory.
    target_directory = Path(target_directory)
    with zipfile.ZipFile(archive_path,"r") as zip_ref:
        zip_ref.extractall(target_directory)
        print(f"Extracted {archive_name} to {target_directory}.")

# Download and extract the testfiles if the setting is enabled. 
if DOWNLOAD_TESTFILES == True:
    print("Testfile download requested.")
    
    # Download the testfiles
    url = ("https://owncloud.gwdg.de/index.php/s/C45WJA0iKdZ6uqn/download")
    download_zip(url, "shotDetection_testfiles", VIDEO_PATH)
else:
     print("No testfile download requested.")

# Download and extract the custom archive if a link is provided.
if CUSTOM_ZIP_LINK != "insert link here":
    print("Custom file download requested.")
 
    # Download the custom video archive.
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, VIDEO_PATH)

Testfile download requested.
shotDetection_testfiles already downloaded.
Extracted shotDetection_testfiles to videos_face.


### Vorbereitung des `InsightFace`-Modells

Je nachdem ob eine Cuda-fähige GPU vorhanden ist oder eine CPU wird ein anderes Modell-Paket zur `Face-Detection`, also der Erkennung von Gesichtern im Bild und zur `Face-Recognition`, also der Zuordnung des Gesichts zu einer bestimmten Person - hier Figur im Film - heruntergeladen und vorbereitet. Die Verwendung unterschiedlicher Modell-Packte für GPU oder CPU gewährleistet, dass die Rechenleistung der jeweiligen Prozessoren möglichst optimal verwendet wird und die besten Erkennungsergebnisse erzielt werden.

Wir haben uns für die Pakete *buffalo_m* für die CPU und *antelopev2* für die GPU entschieden, die bei guter Arbeitsgeschwindigkeit für die Erkennung gute Ergebnisse liefern. Aufgrund der Struktur der `.zip`-Dateien funktioniert der automatische Download innerhalb der Programmbibliothek `InsightFace` nicht für die Modell-Pakete *antelopeV2* und *buffalo_m*, weshalb dieser Download im Jupyter Notebook selbst durchgeführt werden muss. 

In [5]:
# Create model directory
model_path = ".insightface"
os.makedirs(model_path, exist_ok=True)

# Download and apply antelopev2 for cuda-enabled GPUs

# if not cuda:
if cuda:
    url = "https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip"
    download_zip(url, "antelopev2", f"{model_path}/models")
    model = insightface.app.FaceAnalysis(
        name="antelopev2",
        root = f"{model_path}",
        providers=['CUDAExecutionProvider']
    )
    model.prepare(ctx_id=0, det_thresh=FACE_DETECTION_THRESHOLD, det_size=(640, 640))
    print("Using GPU for InsightFace")

# Download and apply buffalo_m for CPUs
else:
    url = "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_m.zip"
    download_zip(url, "buffalo_m", f"{model_path}/models")
    model = insightface.app.FaceAnalysis(
        name="buffalo_m",
        root = model_path,
        providers=['CPUExecutionProvider']
    )
    model.prepare(ctx_id=-1, det_thresh=FACE_DETECTION_THRESHOLD, det_size=(640, 640))
    print("Using CPU for InsightFace")

buffalo_m downloaded.
Extracted buffalo_m to .insightface\models.
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .insightface\models\buffalo_m\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .insightface\models\buffalo_m\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .insightface\models\buffalo_m\det_2.5g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .insightface\models\buffalo_m\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .insightface\models\buffalo_m\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5


### Definition der Hilfsfunktionen
Um eine Wiederholung von identischem Code zu vermeiden, wird einmalig die Funktion *insightface_embed* definiert, welche *Face-Detection* und *Face-Recognition*, also die Erstellung der Embeddings, in sich vereint. Ein Embedding stellt ein Gesicht als Vektor dar, also als eine bestimmte Liste von Zahlen. Diese Vektoren erfassen dabei die spezifischen Merkmale eines Gesichts, also was es von anderen Gesichtern unterscheidet. Der Parameter `FACE_DETECTION_THRESHOLD` kommt hier zum Einsatz um festzulegen, ab welchem Wert ein Gesicht in einem Frame als erkannt gilt. **ToDo: genauer: wie kommt er zum Einsatz? Andere Benennung (einheitlich?) ??**.

In [6]:
def insightface_embed(
    model, # The previously defined models
    image: np.ndarray, # The image or frame
    conf_thresh: float = 0.75, # The face recognition threshold
    ):

    # Raise an error if the image is corrupted.
    if image is None:
        raise FileNotFoundError(f"Could not read image")

    # Detect and represent the faces as embeddings.
    faces = model.get(image)
    
    embeddings = []
    face_metadata = []
    # Enumerate through the detected faces
    for i, face in enumerate(faces):
        # Save the embeddings and metadata if the detection confidence score is higher than the confidence threshold.
        if face.det_score >= conf_thresh:
            embeddings.append(face.normed_embedding)
            face_metadata.append({
                "face_idx": i, # The assigned number of the face in the frame. This could be used to label faces within visualisations.
                "bbox": face.bbox, # The bounding box of the detected face
                "score": float(face.det_score) # The confidence score with which the face has been detected.
            })

    # Return an empty numpy array in the no face has been detected or no face has passed the confidence threshold.
    if len(embeddings) == 0:
        return np.zeros((0, 512), dtype=np.float32), []

    # Stack the embeddings in a numpy array.
    embeddings = np.stack(embeddings).astype(np.float32)

    return embeddings, face_metadata

print("Defined helper function.")

Defined helper function.


## Einlesen der Videos und automatische Figurenerkennung

### Erfassung der Analysematerialen und technischen Videometadaten
Der Video-Ordner wird im nächsten Schritt nach Videodateien, Bilddateien mit Abbildungen der Figuren und csv-Dateien mit Einstellungsannotationen durchsucht. Die Dateipfade sowie die Einstellungen pro Film werden in ein **Dictionary** übertragen. Ein **Dictionary** ist eine Python-Datenstruktur, in der jedem Schlüssel (`key`) ein Wert zugeordnet wird. Im vorliegenden Fall dient der Name eines Videos als Schlüssel. Der zugehörige Wert ist ein weiteres Dictionary, das den Dateipfad, die Figurenabbildungen und die Einstellungsdaten enthält. **ToDo: Prüfen, korrigieren und ggf. ergänzen**


In [7]:
videos = {}
for folder in Path(VIDEO_PATH).iterdir():
    # Exclude Jupyter Book checkpoint folders
    if not folder.is_dir() or "ipynb" in folder.name:
        continue
    video_name = folder.name
    print(f"Opening folder {video_name}.")
    
    # Create per video a dictionary within the larger folder dictionary.
    videos[video_name] = {}
    videos[video_name]["image_files"] = []
    videos[video_name]["video_path"] = None
    videos[video_name]["shots_csv"] = []

    for file in folder.iterdir():
        if file.name.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif')):
            print(f"Found image {file.name}.")
            videos[video_name]["image_files"].append(file)

        elif file.name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv', '.wmv',
                                          '.flv', '.webm', '.mpeg', '.mpg', '.m4v', '.3gp')):
            
            print(f"Found video {file.name}.")
            if not videos[video_name]["video_path"]:
                videos[video_name]["video_path"] = file
            else:
                print(F"More than one video file in folder {video_name}."
                      F"Video {file.name} will not be analysed.")

        elif "_shots" in file.name.lower() and file.name.lower().endswith(".csv"):
           
            print(f"Found shots file {file.name}.")
            if not videos[video_name]["shots_csv"]:
                videos[video_name]["shots_csv"] = file
                shots_csv_found = False
            else:
                print(F"More than one shots.csv in folder {video_name}."
                      F"{file.name} will not be used.")

    if not videos[video_name]["image_files"]:
        print(f"⚠ No image files found in folder {video_name}, analysis cannot proceed.")
    if not videos[video_name]["video_path"]:
        print (f" ⚠ No video files found in folder {video_name}, analysis cannot proceed.")

    print(DIVIDER)


Opening folder Der_geteilte_Himmel_DEFA-Trailer.
Found video Der geteilte Himmel - DEFA-Trailer.mp4.
Found shots file DerGeteilteHimmel_Trailer_shots_0.5.csv.
Found image Manfred01.png.
Found image Manfred02.png.
Found image Manfred03.png.
Found image Manfred04.png.
Found image Rita01.png.
Found image Rita02.png.
Found image Rita03.png.
Found image Rita04.png.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### Erstellung und Prüfung der Referenz-Embeddings
Mittels der zuvor definierten Funktion werden anhand der Dateien mit den Abbildungen der Figuren (meist Screenshots) Embeddings zu den Figuren erstellt, also die charakteristischen Merkmale des Gesichts einer Figur in Zahlenwerten in Form von Vektoren erfasst. Pro Figur kann eine beliebige Anzahl an Abbildungen verwendet werden, aus denen ein durchschnittliches Embedding berechnet wird. Die Namen der Figuren werden den Dateinamen entnommen.

In [8]:
for video_name, video in videos.items():
    print(f"Opening folder {video_name}.")
    video["char_embeddings"] = {}
    for image in video["image_files"]:

        print(image)

        char_name = image.stem[:-2]

        video["char_embeddings"].setdefault(char_name, [])

        print(f"Creating embedding for {char_name} using {image.name}.")

        image_cv2 = cv2.imread(str(image))
        embeddings, face_metadata = insightface_embed(model, image_cv2, 0.6)
        n_embeddings = len(embeddings)
        if n_embeddings == 0:
            print(f"Image {image.name} does not provide a detectable face. Choose a different file.")
        elif n_embeddings > 1:
            print(f"Image {image.name} includes too many faces. Choose a different file.")
        elif n_embeddings == 1:
            video["char_embeddings"][char_name].append(embeddings[0])
            print(f"Image {image.name} is perfect! The embedding was saved for future comparison.")

    char_names = []
    for char_name, embeddings in video["char_embeddings"].items():
        print(char_name)
        char_names.append(char_name)

        embeddings = np.array(embeddings)
        avg_embed = np.mean(embeddings, axis=0)
        norm = np.linalg.norm(avg_embed)
        if norm > 0:
            avg_embed = avg_embed / norm

        video["char_embeddings"][char_name] = avg_embed

    for i in range(len(char_names)):
        for j in range(i + 1, len(char_names)):
            name1 = char_names[i]
            name2 = char_names[j]

            emb1 = video["char_embeddings"][name1]
            emb2 = video["char_embeddings"][name2]

            sim1 = np.dot(emb1, emb2)
            sim2 = np.dot(emb1, emb1)
            sim3 = np.dot(emb2, emb2)

            print(f"{name1} vs {name2}: {sim1:.4f}")
            print(f"{name1} vs {name1}: {sim2:.4f}")
            print(f"{name2} vs {name2}: {sim3:.4f}")

    print(DIVIDER)

Opening folder Der_geteilte_Himmel_DEFA-Trailer.
videos_face\Der_geteilte_Himmel_DEFA-Trailer\Manfred01.png
Creating embedding for Manfred using Manfred01.png.
Image Manfred01.png does not provide a detectable face. Choose a different file.
videos_face\Der_geteilte_Himmel_DEFA-Trailer\Manfred02.png
Creating embedding for Manfred using Manfred02.png.
Image Manfred02.png is perfect! The embedding was saved for future comparison.
videos_face\Der_geteilte_Himmel_DEFA-Trailer\Manfred03.png
Creating embedding for Manfred using Manfred03.png.
Image Manfred03.png is perfect! The embedding was saved for future comparison.
videos_face\Der_geteilte_Himmel_DEFA-Trailer\Manfred04.png
Creating embedding for Manfred using Manfred04.png.
Image Manfred04.png is perfect! The embedding was saved for future comparison.
videos_face\Der_geteilte_Himmel_DEFA-Trailer\Rita01.png
Creating embedding for Rita using Rita01.png.
Image Rita01.png is perfect! The embedding was saved for future comparison.
videos_face

### Anwendung des Modells
Nun wird die eigentliche Analyse der Videodateien ausgeführt. Zunächst werden anhand des Parameters `SKIP_FRAMES` die zu analysierenden Frames aus der Videodatei ausgewählt (siehe auch Abschnitt zu [Festlegung der Parameter](#festlegen-der-parameter)). Der Wert von `SKIP_FRAMES` legt fest, wie  viele Frames nach der Auswertung eines Frames übersprungen werden. Eine Auswertung aller Frames einer Videodatei wäre in der Regel zu zeitintensiv, würde also sehr lange dauern. Wird ein hoher Wert für `SKIP_FRAMES` gewählt, kann jedoch die Genauigkeit der Analyse sinken, da Frames die die zu erkennenden Figuren beieinhalten evtl. übersprungen werden.

Auf die ausgewählten Frames wird anschließend die oben definierte *insightface_embed()* Funktion angewendet. Durch den Wert *detection confidence score* `FACE_DETECTION_THRESHOLD` wird festgelegt, ab wann ein Gesicht in einem Video-Frame als erkannt gilt und weiter verarbeitet wird. Für dieses als erkannt geltendes Gesicht wird ein Embedding erstellt, also die Merkmale in Form von Zahlenwerten als Vektor erfasst. Durch das *Skalarprodukt* erfolgt der Vergleich des Embeddings aus dem Videoframe mit den zuvor erstellten Embeddings aus den Referenzabbildungen der Figuren im Film. Ein auf das Skalarprodukt angewendeter Schwellenwert `CHARACTER_RECOGNITION_THRESHOLD` entscheidet, ob es sich bei dem Gesicht im Videoframe um die zu erkennende Figur handelt. In diesem Fall wird der entsprechende Timecode des Frames im zuvor erstellten Dictionary einer Figur zugeordnet. Jedes Gesicht kann dabei nur einer Figur zugeordnet werden. **ToDo: inhaltlich prüfen, z.B. CHARACTER_RECOGNITION_THRESHOLD etc.**

In [9]:
for video_name, video in videos.items():
    print(f"Opening video {video_name}.")

    video_path =  video["video_path"]
  
    print(f"Analysing video {video_path.name}.")

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Could not open video {video.name}")
        continue

    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    selected_frames = range(0, frame_count, 1 + int(SKIP_FRAMES))

    video["char_frames"] = {}
    video["metadata"] = {
        "video_name"                        : video_path.name,
        "face_detection_threshold"          : FACE_DETECTION_THRESHOLD,
        "character_recognition_threshold"   : CHARACTER_RECOGNITION_THRESHOLD,
        "skip_frames"                       : SKIP_FRAMES,
        "frame_rate"                        : frame_rate,
        "total_frames"                      : frame_count,
        "selected_frames"                   : len(selected_frames),
    }

    print(f"Total frames: {frame_count}")
    print(f"Selected {len(selected_frames)} frames with SKIP_FRAMES = {int(SKIP_FRAMES)}.")

    start_time = time.perf_counter()

    selected_set = set(selected_frames)

    for frame_idx in tqdm(range(frame_count), total=frame_count):

        # Also advances the cap
        ret_boolean, frame = cap.read()
        if not ret_boolean:
                break

        if frame_idx not in selected_set:
            continue

        time_s = frame_idx / frame_rate

        embeddings, face_metadata = insightface_embed(model, frame, float(FACE_DETECTION_THRESHOLD))

        if len(embeddings) > 0:
            for e, embed in enumerate(embeddings):
                best_char = None
                best_dot = -1
                for char_name, char in video["char_embeddings"].items():
                    dot = np.dot(char, embed)

                    if dot > best_dot:
                        best_dot = dot
                        best_char = char_name

                if best_dot >= float(CHARACTER_RECOGNITION_THRESHOLD):
                    video["char_frames"].setdefault(best_char, {})[frame_idx] = {
                        "time_s": time_s,
                        "time_hh_mm_ss": str(datetime.timedelta(seconds=round(time_s))),
                        "dot": best_dot,
                        "score": face_metadata[e]["score"]
                    }

        del embeddings, face_metadata

    cap.release()

    processing_time_seconds = time.perf_counter() - start_time
    elapsed_time_hh_mm_ss = str(datetime.timedelta(seconds=round(processing_time_seconds)))
    video["metadata"]["elapsed_time"] = elapsed_time_hh_mm_ss

    print(f"Analysis took {elapsed_time_hh_mm_ss} for video {video_path.name}.")
    print(DIVIDER)

print(DIVIDER)

Opening video Der_geteilte_Himmel_DEFA-Trailer.
Analysing video Der geteilte Himmel - DEFA-Trailer.mp4.
Total frames: 5225
Selected 1045 frames with SKIP_FRAMES = 4.


  0%|          | 0/5225 [00:00<?, ?it/s]

Analysis took 0:15:53 for video Der geteilte Himmel - DEFA-Trailer.mp4.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### Tabellarische Aufbereitung der Gesichtserkennung
Die Dictionaries werden in Dataframes überführt, also ein spezielles Tabellenformat. Der Grund für die Überführung in Dataframes ist, dass sich aus diesen später sehr einfach die Export-Dateien erstellen lassen. Aus den Ergebnissen der Dataframes werden Metadaten für jede Figur berechnet: Die Anzahl der erkannten Frames mit der Figur, das durchschnittliche Skalarprodukt (also dem Wert für die Ähnlichkeit der verglichenen Embeddings) bei der Erkennung der Figur, den durchschnittlichen Wahrscheinlichkeitswert für die Erkennung des Gesichts in einem Frame für die Figur, die Gesamtzeit der Auftretens einer Figur in Sekunden und in Stunden:Minuten:Sekunden. **ToDo: korrigieren und ergänzen, z.B. Skalarprodukt, score mean**

In [10]:
for video_name, video in videos.items():
    print(f"Opening folder {video_name}.")
    print(DIVIDER)

    video["df_detections"] = []

    for char_name, char in video["char_frames"].items():
        df = pd.DataFrame.from_dict(char, orient='index')
        df["char_name"] = char_name
        df.index.name = "framecount"
        video["df_detections"].append(df)
        
        total_char = len(df)
        dot_mean = str(round(df["dot"].mean(), 2))
        score_mean = str(round(df["score"].mean(), 2))
        print(video["metadata"])
        time_sum_s = total_char  * (SKIP_FRAMES + 1)  * (1 / video["metadata"]["frame_rate"])
        time_sum_hh_mm_ss = str(datetime.timedelta(seconds=round(time_sum_s)))

        video["metadata"].update({
            f"{char_name}_detected_frames"      : total_char,
            f"{char_name}_dot_mean"             : dot_mean,
            f"{char_name}_score_mean"           : score_mean,
            f"{char_name}_time_sum_s"           : time_sum_s,
            f"{char_name}_time_sum_hh_mm_ss"    : time_sum_hh_mm_ss
        })
        print(f"Saved results for {char_name}.")

        print(f"Found {char_name} in {total_char} frames,")
        print(f"with an average face detection score of {score_mean} while using a threshold of {FACE_DETECTION_THRESHOLD},")
        print(f"and an average dot-product of {dot_mean} while using a threshold of {CHARACTER_RECOGNITION_THRESHOLD}.")
        print(f"{char_name} appeared approximately for {time_sum_hh_mm_ss}.")
        print(DIVIDER)
    print(DIVIDER)


Opening folder Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
{'video_name': 'Der geteilte Himmel - DEFA-Trailer.mp4', 'face_detection_threshold': 0.3, 'character_recognition_threshold': 0.3, 'skip_frames': 4, 'frame_rate': 25.0, 'total_frames': 5225, 'selected_frames': 1045, 'elapsed_time': '0:15:53'}
Saved results for Rita.
Found Rita in 391 frames,
with an average face detection score of 0.75 while using a threshold of 0.3,
and an average dot-product of 0.62 while using a threshold of 0.3.
Rita appeared approximately for 0:01:18.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
{'video_name': 'Der geteilte Himmel - DEFA-Trailer.mp4', 'face_detection

### Zusammenfügen der Frames zu Annotationen
Die Daten aus dem Auftreten einer Figur in einzelnen Bild-Frames der Videodatei aus dem ersten Dataframe werden in einem neuen Dataframe zu längeren, zusammenhängenden Annotationen zusammengefügt. Hierfür spielt der in den Parametereinstellungen festgelegte Wert `TOLERANCE_FRAMES` eine Rolle. Bei der Erkennung einer Figur können einzelne Frames fehlen, obwohl die Figur im Bild ist, z.B. da das Gesicht einer Figur kurzzeitig nur schlecht erkennbar ist oder bestimmte Bild-Frames durch den in `SKIP_FRAMES` festgelegten Wert bei der Erkennung übersprungen werden (**siehe Kapitel XX**). Wird die Figur nach dem mit `TOLERANCE_FRAMES` festgelegten Wert an aufeinanderfolgenden Frames wieder im Bild erkannt, wird die Annotation für die Figur in diesem Moment weitergeführt, ansonsten abgebrochen. Dadurch ergeben sich sinnvolle, längere Annotationen zu einzelnen Figuren. **ToDo: Prüfen, ob Erklärung so korrekt**

In [11]:
for video_name, video in videos.items():
    print(f"Opening folder {video_name}.")
    print(DIVIDER)
    video_name = video["video_path"].name
    all_dfs = []

    for df in video["df_detections"]:

        if df.empty:
            continue

        df = df.sort_index()
        char_name = df["char_name"].iloc[0]

        annotations = []
        frames = df.index.tolist()

        start = frames[0]
        prev = frames[0]

        for frame in frames[1:]:
            if frame - prev <= int(SKIP_FRAMES) + int(TOLERANCE_FRAMES) + 1:
                prev = frame
            else:
                annotations.append({
                    "video"             : video_name,
                    "character"         : char_name,
                    "start"             : start,
                    "end"               : prev,
                    "start_s"           : df.loc[start, "time_s"],
                    "end_s"             : df.loc[prev, "time_s"],
                    "start_hh_mm_ss"    : df.loc[start, "time_hh_mm_ss"],
                    "end_hh_mm_ss"      : df.loc[prev, "time_hh_mm_ss"],
                })

                start = frame
                prev = frame

        # final shot
        annotations.append({
            "video"             : video_name,
            "character"         : char_name,
            "start"             : start,
            "end"               : prev,
            "start_s"           : df.loc[start, "time_s"],
            "end_s"             : df.loc[prev, "time_s"],
            "start_hh_mm_ss"    : df.loc[start, "time_hh_mm_ss"],
            "end_hh_mm_ss"      : df.loc[prev, "time_hh_mm_ss"],
        })

        df_annotations = pd.DataFrame(annotations)

        if df_annotations is not None and not df.empty:
            all_dfs.append(df_annotations)

        print(f"Created {len(df_annotations)} annotations with a tolerance of {TOLERANCE_FRAMES} frames.")
        print(df_annotations[["start_hh_mm_ss", "end_hh_mm_ss"]].head(10))
        print(DIVIDER)
        print(DIVIDER)
        
        # Combine all individual dfs on a folder level
        combined_df = pd.concat(
            all_dfs,
            ignore_index=True
        )

        print(combined_df)
        video["df_annotations"] = combined_df
        print(f"Combined all annotation dfs for folder {video_name}.")
    print(DIVIDER)


Opening folder Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Created 11 annotations with a tolerance of 25 frames.
  start_hh_mm_ss end_hh_mm_ss
0        0:00:07      0:00:18
1        0:00:21      0:00:23
2        0:00:40      0:00:49
3        0:00:51      0:00:51
4        0:00:58      0:01:02
5        0:01:03      0:01:12
6        0:01:28      0:01:44
7        0:02:01      0:02:12
8        0:02:17      0:02:25
9        0:02:50      0:02:50
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

---
## Export der Ergebnisse

Die Analyseergebnisse werden als nächstes in mehreren Formaten gespeichert. `.csv`-Dateien eignen sich für Tabellenkalkulation, Statistikprogramme und weitere Python-Auswertungen, etwa zur Visualisierung. Zudem können sie auch  in *OpenRefine* eingelesen werden. Die Verwendung von *OpenRefine* wird in der OER zu <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/1_einf%C3%BChrung.html" class="external-link" target="_blank">OER zu Studentischen Filmen an der Filmuniversität</a> genauer erklärt, ebenso das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/2_import.html#datenformate" class="external-link" target="_blank">csv-Dateiformat</a>.

Das `.eaf`-Format kann direkt in *VIAN* oder anderen Programmen zur Videoannotation geöffnet oder importiert werden, die Daten können dann dort manuell überprüft, bearbeitet und ergänzt werden.

### Export der `.csv` Dateien
Die Annotationen und Metadaten werden als `.csv`-Datei exportiert.

In [12]:
export_directory = Path(EXPORT_PATH)

for video_name, video in videos.items():

    df_detections = pd.concat(video["df_detections"], ignore_index=False)
    df_annotations = video["df_annotations"].copy()
    df_metadata = pd.DataFrame([video["metadata"]])

    # create folder structure
    video_path = export_directory / video_name
    video_path.mkdir(parents=True, exist_ok=True)

    # define file path properly
    frames_path = video_path / f"{video_name}_frames.csv"
    annotations_path = video_path / f"{video_name}_annotations.csv"
    meta_path = video_path / f"{video_name}_meta.csv"


    # overwrite safely
    if frames_path.exists():
        frames_path.unlink()
        print(f"Overwriting {frames_path}.")
    if annotations_path.exists():
        annotations_path.unlink()
        print(f"Overwriting {annotations_path}.")
    if meta_path.exists():
        meta_path.unlink()
        print(f"Overwriting {meta_path}.")

    df_detections.to_csv(frames_path, index=False)
    print(f"Exported {frames_path.name} to {video_path}.")
    print(DIVIDER)

    df_annotations.to_csv(annotations_path, index=False)
    print(f"Exported {annotations_path.name} to {video_path}.")
    print(DIVIDER)

    df_metadata.to_csv(meta_path, index=False)
    print(f"Exported {meta_path.name} to {video_path}.")
    print(DIVIDER)

Exported Der_geteilte_Himmel_DEFA-Trailer_frames.csv to export_face\Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Exported Der_geteilte_Himmel_DEFA-Trailer_annotations.csv to export_face\Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Exported Der_geteilte_Himmel_DEFA-Trailer_meta.csv to export_face\Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### Export der Figuren-Annotationen als `.eaf` Datei

Das `.eaf`-Format speichert Annotationen als Zeitangaben in Millisekunden. Deshalb werden die Start- und Endframes jedes erkannten Figurenauftritts mithilfe der individuellen Bildrate des Videos direkt in Millisekunden umgerechnet. Der Endzeitpunkt liegt unmittelbar nach dem letzten zum Auftreten der Figur gehörenden Frame.

Für jede Figur wird eine Annotationsebene mit dem in den Dateien mit den Referenzabbildungen enthaltenen `Figurennamen` angelegt (**siehe Kapitel xx 4.8?**), die deren Auftreten framegenau darstellt. Zusätzlich wird eine Annotationsbene mit den erkannten Einstellungen und für jede erkannte Figur das Auftreten in einer Einstellung eingefügt. Die Intervalle enthalten anders als gewöhnlich keinen Text, sondern bilden eine zeitliche Segmentierung, die nach dem Import in *VIAN* dort auf ihre korrekte Erkennung hin überprüft und inhaltlich ergänzt werden kann. **[MAYBE CHANGE THIS BEHAVIOUR]**

In [13]:
export_directory = Path(EXPORT_PATH)

for video_name, video in videos.items():
    print(f"Opening folder {video_name}.")
    print(DIVIDER)

    df_annotations = video.get("df_annotations", pd.DataFrame()).copy()
    shots_csv = video.get("shots_csv")

    if df_annotations.empty:
        print(
            f"No annotations found for folder {video_name}. "
            "No .eaf file will be exported."
        )
        print(DIVIDER)
        continue

    eaf = pympi.Eaf()

    char_names = (
        df_annotations["character"]
        .dropna()
        .unique()
    )

    shots = []

    if shots_csv:
        print(f"Including {shots_csv.name}.")
        shots_df = pd.read_csv(shots_csv)

        eaf.add_tier("shots")

        shots = [
            (
                float(row["start_s"]),
                float(row["end_s"])
            )
            for _, row in shots_df.iterrows()
        ]

        for char_name in char_names:
            eaf.add_tier(f"{char_name}_matched")

    for char_name in char_names:
        eaf.add_tier(str(char_name))

    if shots_csv:
        added_shots = set()

        for shot_start, shot_end in shots:
            shot_key = (
                int(round(shot_start * 1000)),
                int(round(shot_end * 1000))
            )

            if shot_key not in added_shots:
                eaf.add_annotation(
                    "shots",
                    shot_key[0],
                    shot_key[1]
                )
                added_shots.add(shot_key)

        added_matches = set()

        for char_name in char_names:
            print(f"Calculating matched shots for {char_name}.")

            char_df = df_annotations[
                df_annotations["character"] == char_name
            ]

            for _, row in char_df.iterrows():
                annotation_start = float(row["start_s"])
                annotation_end = float(row["end_s"])

                for shot_start, shot_end in shots:
                    overlaps = (
                        annotation_start < shot_end
                        and annotation_end > shot_start
                    )

                    if overlaps:
                        match_key = (
                            char_name,
                            int(round(shot_start * 1000)),
                            int(round(shot_end * 1000))
                        )

                        if match_key not in added_matches:
                            eaf.add_annotation(
                                f"{char_name}_matched",
                                match_key[1],
                                match_key[2]
                            )
                            added_matches.add(match_key)

    for char_name in char_names:
        char_df = df_annotations[
            df_annotations["character"] == char_name
        ]

        for _, row in char_df.iterrows():
            start_ms = int(
                round(float(row["start_s"]) * 1000)
            )
            end_ms = int(
                round(float(row["end_s"]) * 1000)
            )

            if end_ms <= start_ms:
                frame_duration_ms = max(
                    1,
                    int(
                        round(
                            1000 / video["metadata"]["frame_rate"]
                        )
                    )
                )
                end_ms = start_ms + frame_duration_ms

            eaf.add_annotation(
                str(char_name),
                start_ms,
                end_ms
            )

    folder_path = export_directory / video_name
    folder_path.mkdir(parents=True, exist_ok=True)

    eaf_path = folder_path / f"{video_name}_annotations.eaf"

    if eaf_path.exists():
        eaf_path.unlink()
        print(f"Overwriting {eaf_path}.")

    eaf.to_file(eaf_path)
    video["eaf_path"] = eaf_path

    print(f"Exported {eaf_path.name} to {folder_path}.")
    print(DIVIDER)

Opening folder Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Including DerGeteilteHimmel_Trailer_shots_0.5.csv.
Calculating matched shots for Rita.
Calculating matched shots for Manfred.
Exported Der_geteilte_Himmel_DEFA-Trailer_annotations.eaf to export_face\Der_geteilte_Himmel_DEFA-Trailer.
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### Herunterladen des *Google Colab* Export-Ordners

Dieser optionale Schritt ist nur für eine Ausführung in *Google Colab* vorgesehen. Der gesamte Exportordner in *Colab* wird zu einer `.zip`-Datei komprimiert und anschließend über die Browseroberfläche heruntergeladen. Der Download ist notwendig, da ansonsten die Erkennungsdaten verloren gehen, da diese nach Beenden der Laufzeit bzw. von *Colab* immer gelöscht werden, ebenso wie alle in Colab zur Analyse hochgeladenen Videodateien.

In einer lokalen *Jupyter Notebook*-Umgebung bleiben die Dateien im anfangs im Codeblock zur [Festlegung der Parameter](#1-festlegen-der-parameter) definierten Export-Ordner erhalten und können von dort entnommen werden. Es ist daher kein zusätzlicher Download-Schritt erforderlich.

In [14]:
if colab:

    export_directory = f"/content/{EXPORT_PATH}"
    zip_path = f"{EXPORT_PATH}.zip"

    !zip -r "{zip_path}" "{export_directory}"

    files.download(zip_path)

    print(f"Downloaded {EXPORT_PATH}.zip.")

else:
    print("Download only works in Google Colab.")

Download only works in Google Colab.
